# Parameter estimation and fitting (Part 1)
This is a Python notebook in which you will practice the concepts learned during the lectures.

## Startup ROOT
Import the ROOT module: this will activate the integration layer with the notebook automatically

In [1]:
import ROOT
import numpy as np
print(ROOT.gROOT.GetVersion())

6.38.00


## Histogram plot and quick fit
Let's start with a simple histogram and `TF1` creation, via a python ftions. Then, we will use the `TF1` to fit the histogram.

In [2]:
# Activate magic for canvas plotting
%jsroot on
# Function creating the histogram
def simple_histogram():
    histo_title="Productivity;C++ Knowledge;Productivity"
    #Create a histogram with 64 bins and an x axis ranging from 0 to 16
    hist=ROOT.TH1F("hist", histo_title, 64, 0, 16)

    # Fill it with random numbers distributed according to a linear function ("pol1")
    hist.FillRandom("pol1", 10000) 
        
    # Change its marker to filled circles
    hist.SetMarkerStyle(ROOT.kFullCircle)
    hist.SetMarkerColor(ROOT.kRed+2)
    # Don't forget to return it (otherwise it will lost scope)
    return hist
    
# Function creating a TF1
def simple_function():
    # creates the function, a robust polinomial 1-degree (predefined: "cheb1")
    f1 = ROOT.TF1("f1", "pol1", 0, 16)
    # Change the line color to something blue
    f1.SetLineColor(ROOT.kBlue)
    # Change to a wider line width as well 
    f1.SetLineWidth(3)
    # Missing somethign?
    return f1

# Call the functions to create the objects
histo = simple_histogram()
f1 = simple_function()

# Fit the histogram with the function
histo.Fit(f1)

# Draw the histogram and the fit result
c1 = ROOT.TCanvas("c1", "Simple histogram", 800, 600)
histo.Draw("E1")
c1.Draw()

****************************************
Minimizer is Linear / Migrad
Chi2                      =      64.5267
NDf                       =           62
p0                        =      15.5176   +/-   1.79483     
p1                        =      17.4655   +/-   0.284005    


Info in <TCanvas::MakeDefCanvas>:  created default TCanvas with name c1
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1


## Using the output of a fit: TFitResult
Including the **`S`** option when calling the `Fit` method will return a `TFitResult` instance, an object providing information about the fit. 

In [3]:
# Create First an empty histogram with 50 bins with range [-10,10]
h1 = ROOT.TH1F("h1", "Gaussian;x;Events", 50, -10, 10)

# Fill with 10000 Gaussian Random numbers (mean=1, sigma=2)
f_gen = ROOT.TF1("f_gen", "gaus", -10, 10)
f_gen.SetParameters(1, 1, 2) # Constant, Mean, Sigma
h1.FillRandom("f_gen", 10000)

# El dibuixem per confirmar que el veiem
c2 = ROOT.TCanvas("c2", "Gaussian Histogram", 800, 600)
h1.SetFillColor(ROOT.kBlue-10) # Color suau per veure les barres
h1.Draw("HIST") 
c2.Draw()

Before Fitting we need to create the fitting function and set its initial parameter values.

In [4]:
# Define here your function
f_fit = ROOT.TF1("f_fit", "gaus", -10, 10)

# Parameters set: 100, 0, 1
f_fit.SetParameters(100, 0, 1)

We fit now the histogram using the Fit method in ROOT. By default the least-square method is used, so if we want to use the likelihood it is needed the option **"L"**. The option **"S"** is used to create a TFitResult object that is returned to the user. If we want to compute the error using `MINOS`, we use the **"E"** option. We want to change also the default minimization engine: we will use `Minuit2`

In [5]:
# Change the minimization engine to Minuit2 (instead of Minuit, which is per default)
ROOT.Math.MinimizerOptions.SetDefaultMinimizer("Minuit2")
# Fit the function with the options L S and E
res = h1.Fit(f_fit, "L S E")

# Plot the the fit with initial parameter values
c3 = ROOT.TCanvas("c3", "Fit", 800, 600)
h1.Draw("E1P")     # representa les dades amb errors
f_fit.Draw("SAME") # repetim la funció de fit
c3.Draw()

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      18.0687
Chi2                      =      36.1374
NDf                       =           47
Edm                       =  2.06319e-06
NCalls                    =           47
Constant                  =      797.977   +/-   9.77406       -9.71942     +9.82932      (Minos) 
Mean                      =     0.977241   +/-   0.0199981     -0.0200105   +0.019987     (Minos) 
Sigma                     =      1.99974   +/-   0.0141452     -0.0140493   +0.0142427    (Minos)  	 (limited)


Note the print out after the fitting, providing information about the minimization process, and the obtained parameters (`Constant`, `Mean` and `Sigma`). You can print-out this table with the `Print` method of `TFitResults` (res)

In [6]:
# Imprimir la taula de TFitResult
res.Print()

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      18.0687
Chi2                      =      36.1374
NDf                       =           47
Edm                       =  2.06319e-06
NCalls                    =           47
Constant                  =      797.977   +/-   9.77406       -9.71942     +9.82932      (Minos) 
Mean                      =     0.977241   +/-   0.0199981     -0.0200105   +0.019987     (Minos) 
Sigma                     =      1.99974   +/-   0.0141452     -0.0140493   +0.0142427    (Minos)  	 (limited)


We can obtain extra information from the `TFitResult` object, such as the correlation matrix of the fit. 
 * [Q]: What is _the correlation matrix of the fit_?

La matriz de correlación indica el grado de relación estadística entre los parámetros obtenidos del ajuste. Los valores pueden ir de -1 a 1. Un valor de 0 significa que son independientes y un valor cercano a -1 ó 1 indica que el problema de minimización no está bien definido.

In [7]:
corrMatrix = res.GetCorrelationMatrix()
corrMatrix.Print()


3x3 matrix is as follows

     |      0    |      1    |      2    |
--------------------------------------------
   0 |          1  -0.0001323     -0.5775 
   1 | -0.0001323           1   0.0002499 
   2 |    -0.5775   0.0002499           1 



Once you fitted your function, you can extract the fitted parameters from the function itself:

In [8]:
print("Fit results:")
print("Gaussian sigma = {0} +/- {1}".format(
    f_fit.GetParameter("Sigma"), 
    f_fit.GetParError(f_fit.GetParNumber("Sigma"))
))

print("Gaussian mean  = {0} +/- {1}".format(
    f_fit.GetParameter("Mean"), 
    f_fit.GetParError(f_fit.GetParNumber("Mean"))
))

Fit results:
Gaussian sigma = 1.9997399010883585 +/- 0.0141452096120529
Gaussian mean  = 0.9772409095464021 +/- 0.019998068311375655
